# N103 · 最长递增子序列与耐心排序

**目标：** 从平方DP推导最小尾值压缩并恢复一条解。

**先修：** N042, N098, N037。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 严格/非严格；tails不等于答案序列；lower/upper bound；前驱恢复。

## 从问题到算法

LIS平方DP保存“以i结尾”的最长长度。耐心排序保存另一种信息：每个长度可达到的最小末值。更小末值不会削弱未来延伸能力，所以可二分更新。tails数组本身不一定是原数组子序列；要恢复答案，必须保存前驱与每个长度的末项位置。信封宽度相等时高度降序，避免被错误同时选中。

## 最小实现

**本章接口：** `lis_quadratic(nums)`、`lis_length(nums)`、`reconstruct_lis(nums)`、`max_envelopes(envelopes)`

In [1]:
from bisect import bisect_left

def lis_quadratic(nums):
    dp=[1]*len(nums)
    for i,x in enumerate(nums):
        dp[i]=1+max((dp[j] for j in range(i) if nums[j]<x),default=0)
    return max(dp,default=0)

def lis_length(nums):
    tails=[]
    for x in nums:
        p=bisect_left(tails,x)
        if p==len(tails): tails.append(x)
        else: tails[p]=x
    return len(tails)

def reconstruct_lis(nums):
    tails=[]; indices=[]; parent=[-1]*len(nums)
    for i,x in enumerate(nums):
        p=bisect_left(tails,x)
        if p: parent[i]=indices[p-1]
        if p==len(tails): tails.append(x); indices.append(i)
        else: tails[p]=x; indices[p]=i
    out=[]; i=indices[-1] if indices else -1
    while i!=-1: out.append(nums[i]); i=parent[i]
    return out[::-1]

def max_envelopes(envelopes):
    return lis_length([h for w,h in sorted(envelopes,key=lambda x:(x[0],-x[1]))])

## 正确性、前提与复杂度

长度相同的递增序列，末值较小者支配较大者。lower_bound找到第一个不小于x的位置，保持严格递增含义；前驱在处理i前记录，保证索引和数值同时递增。相同宽度高度降序后，严格高度LIS不可能选择同宽的两项。

**代价：** 平方版O(n²)时间O(n)空间；二分版O(n log n)时间O(n)空间；恢复额外O(n)前驱。严格递增与非递减的区别是lower_bound/upper_bound，不能随意互换。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[10,9,2,5,3,7,101,18]; tails=[]; rows=[]
for x in a:
    p=bisect_left(tails,x)
    if p==len(tails): tails.append(x)
    else: tails[p]=x
    rows.append((x,p,tails[:]))
show_table(['读入','替换长度下标','最小末值表'],rows)
print('真实恢复序列',reconstruct_lis(a))

读入,替换长度下标,最小末值表
10,0,[10]
9,0,[9]
2,0,[2]
5,1,"[2, 5]"
3,1,"[2, 3]"
7,2,"[2, 3, 7]"
101,3,"[2, 3, 7, 101]"
18,3,"[2, 3, 7, 18]"


真实恢复序列 [2, 3, 7, 18]


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert lis_length([10,9,2,5,3,7,101,18])==4
assert lis_length([2,2,2])==1
assert max_envelopes([[5,4],[6,4],[6,7],[2,3]])==3
assert max_envelopes([[1,1],[1,2],[1,3]])==1
from itertools import product
for n in range(7):
    for a in product(range(3),repeat=n):
        b=reconstruct_lis(a)
        assert len(b)==lis_length(a)==lis_quadratic(a)
        assert all(x<y for x,y in zip(b,b[1:]))
        it=iter(a); assert all(any(x==y for y in it) for x in b)
print('N103: 所有本章断言通过')

N103: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 证明tails按长度保持最小结尾。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 处理信封同宽时排序方向。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 300. Longest Increasing Subsequence（canonical）
- 354. Russian Doll Envelopes（canonical）
- 673. Number of Longest Increasing Subsequence（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。